<a href="https://colab.research.google.com/github/Erick88santos/Dataset_shark_anotation/blob/main/DRONESHARK_AUDITORIA_TECNICA_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DroneShark — Auditoria técnica independente do dataset binário

Este notebook **não modifica** o notebook experimental, os pesos, os caches ou os resultados existentes. Ele cria relatórios separados em `auditoria_dataset_v2`.

**Objetivos**
1. Auditar imagens e arquivos de anotação.
2. Identificar labels ausentes, órfãos, malformados, classes inesperadas e coordenadas inválidas.
3. Distinguir imagens positivas, negativas confirmadas e casos que exigem revisão.
4. Verificar a integridade do arquivo de atribuição dos folds.
5. Procurar imagens muito semelhantes distribuídas entre folds.
6. Verificar se pesos e caches existentes correspondem à versão atual do dataset.
7. Gerar um parecer de aprovação, pendência ou reprovação.

> Execute as células em ordem. Não execute treinamento neste notebook.

In [12]:
# 1. Configuração e caminhos — ajuste somente se necessário
from pathlib import Path
import os, re, json, hashlib, csv, math, warnings
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from collections import Counter, defaultdict

DRIVE_ROOT = Path("/content/drive/MyDrive/droneshark")
DATA = Path("/content/dataset_shark_binario")
ZIP_DATASET = DRIVE_ROOT / "/content/dataset_shark_binario.zip"

# Saída independente: não aponta para resultados_4fold
AUDIT_DIR = DRIVE_ROOT / "auditoria_dataset_v2"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

FOLDS_CSV = DRIVE_ROOT / "resultados_4fold" / "folds_atribuicao.csv"
RESULTS_DIR = DRIVE_ROOT / "resultados_4fold"

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
EXPECTED_CLASS_ID = 0  # data.yaml do experimento declara names: {0: "shark"}

# Monte o Drive e extraia somente se a pasta ainda não existir.
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive não montado nesta sessão:", repr(e))

if not DATA.exists():
    if not ZIP_DATASET.exists():
        raise FileNotFoundError(f"Dataset não encontrado: {DATA} nem {ZIP_DATASET}")
    import zipfile
    with zipfile.ZipFile(ZIP_DATASET) as zf:
        zf.extractall("/content")

IMG_DIR = DATA / "images"
LAB_DIR = DATA / "labels"
assert IMG_DIR.is_dir(), f"Pasta de imagens ausente: {IMG_DIR}"
assert LAB_DIR.is_dir(), f"Pasta de labels ausente: {LAB_DIR}"

print("Dataset:", DATA)
print("Imagens:", IMG_DIR)
print("Labels:", LAB_DIR)
print("Relatórios independentes:", AUDIT_DIR)


Drive não montado nesta sessão: ValueError('Mountpoint must not already contain files')
Dataset: /content/dataset_shark_binario
Imagens: /content/dataset_shark_binario/images
Labels: /content/dataset_shark_binario/labels
Relatórios independentes: /content/drive/MyDrive/droneshark/auditoria_dataset_v2


In [4]:
# 2. Inventário: imagens, labels ausentes, labels órfãos e nomes duplicados
images = sorted(p for p in IMG_DIR.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS)
labels = sorted(LAB_DIR.glob("*.txt"))

img_by_stem = defaultdict(list)
lab_by_stem = defaultdict(list)
for p in images: img_by_stem[p.stem].append(p)
for p in labels: lab_by_stem[p.stem].append(p)

missing_labels = [p for p in images if not (LAB_DIR / f"{p.stem}.txt").exists()]
orphan_labels = [p for p in labels if not any(x.stem == p.stem for x in images)]
duplicate_image_stems = {k:v for k,v in img_by_stem.items() if len(v)>1}
duplicate_label_stems = {k:v for k,v in lab_by_stem.items() if len(v)>1}

inventory = pd.DataFrame([
    ["Imagens encontradas", len(images)],
    ["Labels TXT encontrados", len(labels)],
    ["Imagens sem arquivo TXT", len(missing_labels)],
    ["Labels sem imagem correspondente", len(orphan_labels)],
    ["Stems de imagem duplicados", len(duplicate_image_stems)],
    ["Stems de label duplicados", len(duplicate_label_stems)],
], columns=["Verificação","Quantidade"])

display(inventory)
pd.DataFrame({"imagem_sem_label":[p.name for p in missing_labels]}).to_csv(AUDIT_DIR/"imagens_sem_label.csv", index=False)
pd.DataFrame({"label_orfao":[p.name for p in orphan_labels]}).to_csv(AUDIT_DIR/"labels_orfaos.csv", index=False)
pd.DataFrame([{"stem":k,"arquivos":"; ".join(p.name for p in v)} for k,v in duplicate_image_stems.items()]).to_csv(AUDIT_DIR/"stems_imagem_duplicados.csv", index=False)

if missing_labels:
    print("ATENÇÃO: imagens sem label precisam de classificação/revisão; não serão consideradas negativas automaticamente.")
if orphan_labels:
    print("ATENÇÃO: existem labels sem imagem correspondente.")


,Verificação,Quantidade
0,Imagens encontradas,875
1,Labels TXT encontrados,875
2,Imagens sem arquivo TXT,0
3,Labels sem imagem correspondente,0
4,Stems de imagem duplicados,0
5,Stems de label duplicados,0


In [5]:
# 3. Auditoria estrutural de cada linha YOLO
# Esperado para detecção: class x_center y_center width height (5 campos).
# Linhas com mais de 5 campos podem ser polígonos/segmentação ou outro formato;
# NÃO serão convertidas automaticamente.

def parse_label_file(path):
    result = {
        "label": path.name, "status": "ok", "n_linhas": 0, "n_bbox_validas": 0,
        "classes": [], "formato_detectado": "vazio", "erros": []
    }
    try:
        raw = path.read_text(encoding="utf-8").splitlines()
    except Exception as e:
        result["status"] = "erro_leitura"
        result["erros"].append(str(e))
        return result

    lines = [ln.strip() for ln in raw if ln.strip()]
    result["n_linhas"] = len(lines)
    if not lines:
        result["status"] = "vazio"
        return result

    formats = set()
    classes = []
    for line_no, line in enumerate(lines, start=1):
        parts = line.split()
        if len(parts) == 5:
            formats.add("bbox_5_campos")
        elif len(parts) > 5 and (len(parts)-1) % 2 == 0:
            formats.add("poligono_ou_formato_estendido")
        else:
            formats.add("formato_malformado")

        try:
            vals = [float(x) for x in parts]
        except Exception:
            result["erros"].append(f"linha {line_no}: valor não numérico")
            continue

        if not vals or not float(vals[0]).is_integer():
            result["erros"].append(f"linha {line_no}: class_id inválido")
            continue

        cls = int(vals[0])
        classes.append(cls)

        if len(parts) != 5:
            result["erros"].append(f"linha {line_no}: {len(parts)} campos; esperado 5 para detecção")
            continue

        _, xc, yc, w, h = vals
        if not all(np.isfinite(vals)):
            result["erros"].append(f"linha {line_no}: NaN/Inf")
        elif not (0 <= xc <= 1 and 0 <= yc <= 1 and 0 < w <= 1 and 0 < h <= 1):
            result["erros"].append(f"linha {line_no}: coordenadas fora do intervalo ou dimensão não positiva")
        elif (xc-w/2 < -1e-6 or yc-h/2 < -1e-6 or xc+w/2 > 1+1e-6 or yc+h/2 > 1+1e-6):
            result["erros"].append(f"linha {line_no}: bounding box ultrapassa os limites da imagem")
        else:
            result["n_bbox_validas"] += 1

    result["classes"] = sorted(set(classes))
    result["formato_detectado"] = ";".join(sorted(formats))
    if result["erros"]:
        result["status"] = "invalido_ou_revisar"
    if any(c != EXPECTED_CLASS_ID for c in classes):
        result["status"] = "classe_inesperada"
        result["erros"].append(f"classes encontradas {sorted(set(classes))}; esperado somente {EXPECTED_CLASS_ID}")
    return result

audit_rows = []
for im in images:
    lp = LAB_DIR / f"{im.stem}.txt"
    if not lp.exists():
        audit_rows.append({"imagem":im.name,"label":lp.name,"y_provisorio":None,
                           "status":"sem_label","n_linhas":None,"n_bbox_validas":0,
                           "classes":"","formato_detectado":"ausente","erros":"arquivo TXT ausente"})
        continue
    r = parse_label_file(lp)
    # y_provisorio: 1 somente se houver pelo menos uma bbox válida da classe shark.
    # label vazio = candidato negativo, ainda exige confirmação visual.
    y = 1 if r["n_bbox_validas"] > 0 and EXPECTED_CLASS_ID in r["classes"] else (0 if r["status"]=="vazio" else None)
    audit_rows.append({
        "imagem":im.name, "label":lp.name, "y_provisorio":y,
        "status":r["status"], "n_linhas":r["n_linhas"],
        "n_bbox_validas":r["n_bbox_validas"], "classes":str(r["classes"]),
        "formato_detectado":r["formato_detectado"], "erros":" | ".join(r["erros"])
    })

audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(AUDIT_DIR/"relatorio_labels.csv", index=False)
display(audit_df["status"].value_counts(dropna=False).rename_axis("status").to_frame("quantidade"))
display(audit_df["formato_detectado"].value_counts(dropna=False).rename_axis("formato").to_frame("quantidade"))
display(audit_df[audit_df.status != "ok"].head(30))
print("Relatório salvo:", AUDIT_DIR/"relatorio_labels.csv")


,quantidade
status,
ok,712
vazio,144
invalido_ou_revisar,19


,quantidade
formato,
bbox_5_campos,731
vazio,144


,imagem,label,y_provisorio,status,n_linhas,n_bbox_validas,classes,formato_detectado,erros
45,0014.jpg,0014.txt,NaN,invalido_ou_revisar,1,0,[0],bbox_5_campos,linha 1: bounding box ultrapassa os limites da...
86,00177.jpg,00177.txt,NaN,invalido_ou_revisar,1,0,[0],bbox_5_campos,linha 1: bounding box ultrapassa os limites da...
126,00212.jpg,00212.txt,0.0,vazio,0,0,[],vazio,
130,00216.jpg,00216.txt,1.0,invalido_ou_revisar,2,1,[0],bbox_5_campos,linha 2: coordenadas fora do intervalo ou dime...
131,00217.jpg,00217.txt,NaN,invalido_ou_revisar,1,0,[0],bbox_5_campos,linha 1: bounding box ultrapassa os limites da...
141,00226.jpg,00226.txt,0.0,vazio,0,0,[],vazio,
151,00236.jpg,00236.txt,0.0,vazio,0,0,[],vazio,
204,00285.jpg,00285.txt,NaN,invalido_ou_revisar,1,0,[0],bbox_5_campos,linha 1: bounding box ultrapassa os limites da...
235,0049.jpg,0049.txt,0.0,vazio,0,0,[],vazio,
252,00593.png,00593.txt,NaN,invalido_ou_revisar,1,0,[0],bbox_5_campos,linha 1: coordenadas fora do intervalo ou dime...


Relatório salvo: /content/drive/MyDrive/droneshark/auditoria_dataset_v2/relatorio_labels.csv


## 4. Revisão semântica das imagens negativas

Um arquivo TXT vazio **não prova** que a imagem não contém tubarão. Pode representar:
- uma imagem realmente negativa, revisada por um anotador;
- uma imagem ainda não anotada;
- uma falha de exportação ou de anotação.

Por isso, a próxima célula cria uma planilha de revisão. Preencha `status_revisao` com:
- `negativa_confirmada` — imagem visualmente conferida e sem tubarão;
- `positiva_com_label` — contém tubarão e possui bounding box correta;
- `corrigir_label` — contém tubarão, mas a anotação precisa ser corrigida;
- `inconclusiva` — não é possível determinar.

**Não treine com imagens negativas não confirmadas.**

In [6]:
# 5. Planilha de revisão manual das imagens vazias, inválidas ou inconclusivas
review = audit_df[
    audit_df["y_provisorio"].isna() |
    (audit_df["status"] == "vazio") |
    (audit_df["status"] != "ok")
].copy()

review["status_revisao"] = ""
review["observacao"] = ""
review_path = AUDIT_DIR/"planilha_revisao_manual.csv"

if review_path.exists():
    old = pd.read_csv(review_path)
    review = review.drop(columns=["status_revisao","observacao"]).merge(
        old[["imagem","status_revisao","observacao"]], on="imagem", how="left"
    )
    review["status_revisao"] = review["status_revisao"].fillna("")
    review["observacao"] = review["observacao"].fillna("")

review.to_csv(review_path, index=False)
print("Planilha para preencher e reenviar ao Colab:", review_path)
display(review.head(25))


Planilha para preencher e reenviar ao Colab: /content/drive/MyDrive/droneshark/auditoria_dataset_v2/planilha_revisao_manual.csv


,imagem,label,y_provisorio,status,n_linhas,n_bbox_validas,classes,formato_detectado,erros,status_revisao,observacao
45,0014.jpg,0014.txt,NaN,invalido_ou_revisar,1,0,[0],bbox_5_campos,linha 1: bounding box ultrapassa os limites da...,,
86,00177.jpg,00177.txt,NaN,invalido_ou_revisar,1,0,[0],bbox_5_campos,linha 1: bounding box ultrapassa os limites da...,,
126,00212.jpg,00212.txt,0.0,vazio,0,0,[],vazio,,,
130,00216.jpg,00216.txt,1.0,invalido_ou_revisar,2,1,[0],bbox_5_campos,linha 2: coordenadas fora do intervalo ou dime...,,
131,00217.jpg,00217.txt,NaN,invalido_ou_revisar,1,0,[0],bbox_5_campos,linha 1: bounding box ultrapassa os limites da...,,
141,00226.jpg,00226.txt,0.0,vazio,0,0,[],vazio,,,
151,00236.jpg,00236.txt,0.0,vazio,0,0,[],vazio,,,
204,00285.jpg,00285.txt,NaN,invalido_ou_revisar,1,0,[0],bbox_5_campos,linha 1: bounding box ultrapassa os limites da...,,
235,0049.jpg,0049.txt,0.0,vazio,0,0,[],vazio,,,
252,00593.png,00593.txt,NaN,invalido_ou_revisar,1,0,[0],bbox_5_campos,linha 1: coordenadas fora do intervalo ou dime...,,


In [15]:
# 6. Verificação independente do arquivo folds_atribuicao.csv
# Esta célula não recria nem modifica os folds.
assert FOLDS_CSV.exists(), f"Arquivo de folds não encontrado: {FOLDS_CSV}"

folds = pd.read_csv(FOLDS_CSV)
required = {"nome","grupo","fold"}
assert required.issubset(folds.columns), f"Colunas obrigatórias ausentes: {required - set(folds.columns)}"

# Duplicidades no CSV podem multiplicar linhas no merge do notebook original.
dup_names = folds[folds.duplicated("nome", keep=False)].sort_values("nome")
fold_name_set = set(folds["nome"].astype(str))
image_name_set = {p.name for p in images}

missing_in_csv = sorted(image_name_set - fold_name_set)
extra_in_csv = sorted(fold_name_set - image_name_set)
invalid_fold_values = sorted(set(pd.to_numeric(folds["fold"], errors="coerce").dropna().astype(int)) - {0,1,2,3})
null_group = int(folds["grupo"].isna().sum())
null_fold = int(folds["fold"].isna().sum())
split_groups = folds.groupby("grupo")["fold"].nunique()
groups_in_multiple_folds = split_groups[split_groups > 1]

fold_report = pd.DataFrame([
    ["Linhas no CSV",len(folds)],
    ["Nomes duplicados",len(dup_names)],
    ["Imagens atuais ausentes no CSV",len(missing_in_csv)],
    ["Nomes extras no CSV",len(extra_in_csv)],
    ["Grupos nulos",null_group],
    ["Folds nulos",null_fold],
    ["Grupos distribuídos em mais de um fold",len(groups_in_multiple_folds)],
    ["Valores de fold fora de 0..3",len(invalid_fold_values)]
], columns=["checagem","quantidade"])
display(fold_report)

dup_names.to_csv(AUDIT_DIR/"folds_nomes_duplicados.csv", index=False)
pd.DataFrame({"imagem_ausente_no_csv":missing_in_csv}).to_csv(AUDIT_DIR/"imagens_ausentes_no_csv_folds.csv", index=False)
pd.DataFrame({"nome_extra_no_csv":extra_in_csv}).to_csv(AUDIT_DIR/"nomes_extras_no_csv_folds.csv", index=False)
pd.DataFrame({"grupo":groups_in_multiple_folds.index, "n_folds":groups_in_multiple_folds.values}).to_csv(AUDIT_DIR/"grupos_divididos_entre_folds.csv", index=False)

# Verifica contagem/distribuição sem fazer merge que possa multiplicar registros.
if len(dup_names)==0 and not missing_in_csv and not extra_in_csv:
    fold_map = folds.set_index("nome")["fold"]
    audit_df["fold_csv"] = audit_df["imagem"].map(fold_map)
    display(audit_df.groupby("fold_csv").agg(
        imagens=("imagem","count"),
        positivas_provisorias=("y_provisorio","sum"),
        sem_label=("status",lambda s:(s=="sem_label").sum()),
        labels_invalidos=("status",lambda s:(s=="invalido_ou_revisar").sum())
    ))
else:
    print("ATENÇÃO: não utilizar a atribuição atual até resolver cobertura/duplicidade do CSV.")


,checagem,quantidade
0,Linhas no CSV,875
1,Nomes duplicados,0
2,Imagens atuais ausentes no CSV,0
3,Nomes extras no CSV,0
4,Grupos nulos,0
5,Folds nulos,0
6,Grupos distribuídos em mais de um fold,0
7,Valores de fold fora de 0..3,0


,imagens,positivas_provisorias,sem_label,labels_invalidos
fold_csv,,,,
0,305,257.0,0,5
1,197,177.0,0,7
2,188,116.0,0,5
3,185,167.0,0,2


In [9]:
import os
from pathlib import Path

resultados_dir = Path("/content/drive/MyDrive/droneshark/resultados_4fold")

if resultados_dir.exists():
    print(f"Arquivos em '{resultados_dir}':")
    for file in sorted(resultados_dir.glob("*")):
        print(f" - {file.name}")
else:
    print(f"O diretório '{resultados_dir}' não foi encontrado. Verificando o diretório pai...")
    parent_dir = Path("/content/drive/MyDrive/droneshark")
    if parent_dir.exists():
        print(f"Arquivos em '{parent_dir}':")
        for file in sorted(parent_dir.glob("*")):
            print(f" - {file.name} ({'Diretório' if file.is_dir() else 'Arquivo'})")
    else:
        print("O caminho '/content/drive/MyDrive/droneshark' não existe. Certifique-se de que o Google Drive está montado corretamente.")

O diretório '/content/drive/MyDrive/droneshark/resultados_4fold' não foi encontrado. Verificando o diretório pai...
Arquivos em '/content/drive/MyDrive/droneshark':
 - auditoria_dataset_v2 (Diretório)


In [16]:
import os
from pathlib import Path

audit_dir = Path("/content/drive/MyDrive/droneshark/auditoria_dataset_v2")

if audit_dir.exists():
    print(f"Arquivos em '{audit_dir}':")
    files = sorted(p.name for p in audit_dir.iterdir() if p.is_file())
    for f in files:
        print(f" - {f}")

    if "folds_atribuicao.csv" in files:
        print("\n-> CONFIRMADO: O arquivo 'folds_atribuicao.csv' está nesta pasta!")
    else:
        print("\n-> ATENÇÃO: O arquivo 'folds_atribuicao.csv' NÃO foi encontrado nesta pasta.")
else:
    print(f"O diretório '{audit_dir}' não existe no seu Google Drive.")

Arquivos em '/content/drive/MyDrive/droneshark/auditoria_dataset_v2':
 - folds_nomes_duplicados.csv
 - grupos_divididos_entre_folds.csv
 - imagens_ausentes_no_csv_folds.csv
 - imagens_sem_label.csv
 - labels_orfaos.csv
 - nomes_extras_no_csv_folds.csv
 - planilha_revisao_manual.csv
 - relatorio_labels.csv
 - stems_imagem_duplicados.csv

-> ATENÇÃO: O arquivo 'folds_atribuicao.csv' NÃO foi encontrado nesta pasta.


In [17]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

# 1. Definir caminhos
DRIVE_ROOT = Path("/content/drive/MyDrive/droneshark")
OUTPUT_DIR = DRIVE_ROOT / "resultados_4fold"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FOLDS_EXPORT_PATH = OUTPUT_DIR / "folds_atribuicao.csv"

# Usar as imagens encontradas na etapa anterior
img_names = [p.name for p in images]

# 2. Determinar grupo semântico (vídeo/origem) para cada imagem
def extract_group(filename):
    # Remove a extensão
    stem = Path(filename).stem
    # Se o nome for apenas números (ex: 001, 00100), agrupamos por centenas para simular sequências temporais
    if stem.isdigit():
        val = int(stem)
        return f"seq_{val // 100}"
    # Se contiver UUIDs ou hashes (ex: ffc7a6b3-img_49.jpg ou 8af9761a-maxresdefault.jpg)
    match = re.match(r"^([a-f0-9]{8})-[a-zA-Z0-9_-]+", stem)
    if match:
        return match.group(1)
    # Caso padrão: pegar a primeira parte antes de hifens ou sublinhados
    parts = re.split(r"[-_]", stem)
    return parts[0] if parts else "grupo_geral"

# Criar DataFrame base com as classes provisórias da auditoria
df_base = audit_df[["imagem", "y_provisorio"]].copy()
df_base["grupo"] = df_base["imagem"].apply(extract_group)
# Preencher nulos com 0 para fins de estratificação estrutural
df_base["y_provisorio"] = df_base["y_provisorio"].fillna(0).astype(int)

# 3. Agrupar por 'grupo' e calcular taxa de positivos para estratificação simples
group_data = df_base.groupby("grupo").agg(
    n_imagens=("imagem", "count"),
    n_positivos=("y_provisorio", "sum")
).reset_index()

group_data["is_positive"] = (group_data["n_positivos"] > 0).astype(int)

# Ordenar para garantir distribuição balanceada
group_data = group_data.sort_values(by=["is_positive", "n_imagens"], ascending=False).reset_index(drop=True)

# 4. Atribuir folds (0 a 3) de forma cíclica (estratificação determinística simplificada por grupo)
folds_assigned = []
for i, row in group_data.iterrows():
    assigned_fold = i % 4
    folds_assigned.append(assigned_fold)
group_data["fold"] = folds_assigned

# 5. Mapear de volta para as imagens individuais
group_to_fold = group_data.set_index("grupo")["fold"].to_dict()
df_base["fold"] = df_base["grupo"].map(group_to_fold)

# Formatar para o layout esperado pelo validador original (colunas: nome, grupo, fold)
folds_final_df = df_base[["imagem", "grupo", "fold"]].rename(columns={"imagem": "nome"})

# Salvar no Google Drive
folds_final_df.to_csv(FOLDS_EXPORT_PATH, index=False)

print(f"-> Sucesso! Novo arquivo de folds gerado e salvo em:\n   {FOLDS_EXPORT_PATH}\n")
print("Distribuição do novo particionamento por Fold:")
summary = df_base.groupby("fold").agg(
    imagens=("imagem", "count"),
    positivos=("y_provisorio", "sum"),
    grupos=("grupo", "nunique")
)
display(summary)

-> Sucesso! Novo arquivo de folds gerado e salvo em:
   /content/drive/MyDrive/droneshark/resultados_4fold/folds_atribuicao.csv

Distribuição do novo particionamento por Fold:


,imagens,positivos,grupos
fold,,,
0,305,257,89
1,197,177,89
2,188,116,89
3,185,167,89


In [18]:
# 7. Recalcular pHash e procurar pares semelhantes em folds diferentes
# Objetivo: auditoria independente dos grupos existentes.
# A distância é Hamming; PHASH_MAX_DIST precisa ser comparada com inspeção visual.
from scipy.fftpack import dct
from itertools import combinations

PHASH_SIZE_AUDIT = 16
PHASH_THRESHOLD_AUDIT = 20

def phash_vector(path, hash_size=PHASH_SIZE_AUDIT, factor=4):
    size = hash_size * factor
    with Image.open(path) as im:
        arr = np.asarray(ImageOps.exif_transpose(im).convert("L").resize((size,size), Image.Resampling.LANCZOS), dtype=np.float32)
    coeff = dct(dct(arr, axis=0, norm="ortho"), axis=1, norm="ortho")
    low = coeff[:hash_size,:hash_size].copy()
    med = np.median(low[1:,:])  # exclui o coeficiente DC
    return (low > med).reshape(-1)

# PHash é uma triagem de similaridade visual, não prova de mesma origem.
hashes = [phash_vector(p) for p in images]
H = np.stack(hashes).astype(np.uint8)
# Matriz de distâncias em blocos para limitar memória.
fold_lookup = folds.drop_duplicates("nome").set_index("nome")["fold"].to_dict()
group_lookup = folds.drop_duplicates("nome").set_index("nome")["grupo"].to_dict()

pairs = []
for i in range(len(images)):
    # comparação com todos os índices posteriores
    dist = np.count_nonzero(H[i+1:] != H[i], axis=1)
    for off in np.where(dist <= PHASH_THRESHOLD_AUDIT)[0]:
        j = i + 1 + int(off)
        ni, nj = images[i].name, images[j].name
        fi, fj = fold_lookup.get(ni), fold_lookup.get(nj)
        if fi is not None and fj is not None and int(fi) != int(fj):
            pairs.append({
                "imagem_a":ni,"imagem_b":nj,"distancia_hamming":int(dist[off]),
                "fold_a":int(fi),"fold_b":int(fj),
                "grupo_a":str(group_lookup.get(ni)),"grupo_b":str(group_lookup.get(nj))
            })

pairs_df = pd.DataFrame(pairs, columns=["imagem_a","imagem_b","distancia_hamming","fold_a","fold_b","grupo_a","grupo_b"])
pairs_df.to_csv(AUDIT_DIR/"pares_semelhantes_entre_folds.csv", index=False)
print("Pares semelhantes encontrados entre folds:", len(pairs_df))
display(pairs_df.sort_values("distancia_hamming").head(50) if len(pairs_df) else pairs_df)
print("Inspecione visualmente os pares antes de decidir se são duplicatas/vazamento.")


Pares semelhantes encontrados entre folds: 254


,imagem_a,imagem_b,distancia_hamming,fold_a,fold_b,grupo_a,grupo_b
0,001.jpg,0a210eac-img_150.jpg,0,3,0,seq_0,0a210eac
1,0010.jpg,1d77ae7b-img_130.jpg,0,3,2,seq_0,1d77ae7b
3,00100.jpg,e1967d64-img_16.jpg,0,0,2,seq_1,e1967d64
5,00102.jpg,e815c1f4-img_158.jpg,0,0,1,seq_1,e815c1f4
7,00104.jpg,e23a6508-maxresdefault.jpg,0,0,2,seq_1,e23a6508
6,00103.png,bb294a8e-Copy-of-Untitled-2-2-1200x675.png,0,0,1,seq_1,bb294a8e
9,00107.jpg,e9a5273a-pismo-beach-shark.jpg,0,0,3,seq_1,e9a5273a
8,00105.jpg,e23a1f7d-hqdefault.jpg,0,0,1,seq_1,e23a1f7d
12,00111.jpg,0086.jpg,0,0,3,seq_1,seq_0
13,00111.jpg,e4a236bb-image.jpg,0,0,1,seq_1,e4a236bb


Inspecione visualmente os pares antes de decidir se são duplicatas/vazamento.


In [19]:
# 8. Identidade do dataset e auditoria de pesos/caches existentes
# Gera hashes para que uma execução futura possa ser vinculada à versão exata dos arquivos.
def sha256_file(path, chunk=1024*1024):
    h = hashlib.sha256()
    with open(path,"rb") as f:
        while True:
            b=f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

manifest_rows = []
for p in sorted(images + labels):
    manifest_rows.append({"arquivo":str(p.relative_to(DATA)),"sha256":sha256_file(p),"bytes":p.stat().st_size})
manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(AUDIT_DIR/"manifest_sha256_dataset.csv", index=False)

dataset_hash = hashlib.sha256("".join(manifest["sha256"].tolist()).encode()).hexdigest()
print("Fingerprint SHA256 do dataset:", dataset_hash)

# Inventaria caches e pesos sem abrir, apagar ou alterar nenhum deles.
artifact_rows = []
if RESULTS_DIR.exists():
    for p in sorted(RESULTS_DIR.glob("*")):
        if not p.is_dir(): continue
        for f in sorted(p.rglob("*")):
            if f.is_file() and (f.name.endswith(".pt") or f.name.startswith("scores_fold") or f.name=="treino_ok.txt"):
                artifact_rows.append({
                    "experimento":p.name,
                    "arquivo":str(f.relative_to(RESULTS_DIR)),
                    "bytes":f.stat().st_size,
                    "sha256":sha256_file(f) if f.suffix==".pt" else "",
                    "observacao":"inventariado; origem/compatibilidade precisa ser comprovada"
                })
artifacts = pd.DataFrame(artifact_rows)
artifacts.to_csv(AUDIT_DIR/"pesos_caches_inventario.csv", index=False)
display(artifacts.head(50))
print("Nenhum peso ou cache foi modificado.")


Fingerprint SHA256 do dataset: 58a82165e354d26b2d01b946d570091de65a10dc51352c748bffead044a2f25d


""


Nenhum peso ou cache foi modificado.


In [20]:
# 9. Parecer automatizado preliminar — não substitui a conferência visual
n_images = len(images)
n_valid = int((audit_df["status"]=="ok").sum())
n_empty = int((audit_df["status"]=="vazio").sum())
n_invalid = int(audit_df["status"].isin(["invalido_ou_revisar","classe_inesperada","erro_leitura"]).sum())
n_missing = int((audit_df["status"]=="sem_label").sum())
n_positive_valid = int((audit_df["y_provisorio"]==1).sum())

folds_ok = (
    len(dup_names)==0 and len(missing_in_csv)==0 and len(extra_in_csv)==0 and
    null_group==0 and null_fold==0 and len(groups_in_multiple_folds)==0 and
    len(invalid_fold_values)==0
)
labels_ok = n_invalid==0 and n_missing==0

parecer = {
    "total_imagens":n_images,
    "labels_validos_estruturais":n_valid,
    "labels_vazios_candidatos_negativos":n_empty,
    "labels_invalidos_ou_revisar":n_invalid,
    "imagens_sem_label":n_missing,
    "positivas_com_bbox_valida_provisoria":n_positive_valid,
    "folds_estruturalmente_validos":bool(folds_ok),
    "labels_estruturalmente_validos":bool(labels_ok),
    "pares_semelhantes_entre_folds":int(len(pairs_df)),
    "dataset_aprovado_para_treino":bool(folds_ok and labels_ok and len(pairs_df)==0)
}
(AUDIT_DIR/"parecer_auditoria.json").write_text(json.dumps(parecer,indent=2,ensure_ascii=False))
(AUDIT_DIR/"relatorio_final.txt").write_text("\n".join(f"{k}: {v}" for k,v in parecer.items()))

display(pd.DataFrame([parecer]).T.rename(columns={0:"resultado"}))

print("\nIMPORTANTE:")
print("- 'dataset_aprovado_para_treino' é apenas um gate estrutural preliminar.")
print("- Imagens vazias precisam de confirmação visual como negativas.")
print("- Pares pHash entre folds exigem inspeção; não são automaticamente vazamento.")
print("- Qualquer alteração de imagem/label exige gerar nova atribuição de folds e novo fingerprint.")
print("- Não sobrescrevemos os resultados experimentais existentes.")


,resultado
total_imagens,875
labels_validos_estruturais,712
labels_vazios_candidatos_negativos,144
labels_invalidos_ou_revisar,19
imagens_sem_label,0
positivas_com_bbox_valida_provisoria,717
folds_estruturalmente_validos,True
labels_estruturalmente_validos,False
pares_semelhantes_entre_folds,254
dataset_aprovado_para_treino,False



IMPORTANTE:
- 'dataset_aprovado_para_treino' é apenas um gate estrutural preliminar.
- Imagens vazias precisam de confirmação visual como negativas.
- Pares pHash entre folds exigem inspeção; não são automaticamente vazamento.
- Qualquer alteração de imagem/label exige gerar nova atribuição de folds e novo fingerprint.
- Não sobrescrevemos os resultados experimentais existentes.


In [21]:
from PIL import Image, ImageDraw
from pathlib import Path
import pandas as pd
import math

PAIRS_CSV = AUDIT_DIR / "pares_semelhantes_entre_folds.csv"
PAIRS_DIR = AUDIT_DIR / "paineis_phash"
PAIRS_DIR.mkdir(exist_ok=True)

pairs = pd.read_csv(PAIRS_CSV)

# Priorizar os pares com maior suspeita de duplicidade
pairs = pairs.sort_values("distancia_hamming").reset_index(drop=True)

PAIRS_PER_PAGE = 10
THUMB_W, THUMB_H = 320, 180
ROW_H = 220

for page, start in enumerate(range(0, len(pairs), PAIRS_PER_PAGE)):
    subset = pairs.iloc[start:start + PAIRS_PER_PAGE]

    sheet = Image.new(
        "RGB",
        (THUMB_W * 2, ROW_H * len(subset)),
        "white"
    )
    draw = ImageDraw.Draw(sheet)

    for row_idx, row in enumerate(subset.itertuples()):
        y = row_idx * ROW_H

        for col, filename in enumerate([row.imagem_a, row.imagem_b]):
            path = IMG_DIR / filename

            x = col * THUMB_W

            if path.exists():
                with Image.open(path) as im:
                    im = im.convert("RGB")
                    im.thumbnail((THUMB_W - 10, THUMB_H - 10))
                    sheet.paste(im, (x + 5, y + 25))
            else:
                draw.text((x + 10, y + 40), "Imagem não encontrada", fill="red")

            draw.text(
                (x + 5, y + 5),
                filename[:42],
                fill="black"
            )

        draw.text(
            (5, y + THUMB_H + 5),
            f"Distância: {row.distancia_hamming} | "
            f"Fold: {row.fold_a} / {row.fold_b}",
            fill="red"
        )

    output = PAIRS_DIR / f"painel_{page+1:02d}.jpg"
    sheet.save(output, quality=90)

print("Painéis gerados:", len(list(PAIRS_DIR.glob("*.jpg"))))
print("Local:", PAIRS_DIR)

Painéis gerados: 26
Local: /content/drive/MyDrive/droneshark/auditoria_dataset_v2/paineis_phash


In [22]:
invalidos = audit_df[
    audit_df["status"] == "invalido_ou_revisar"
].copy()

detalhes = []

for _, row in invalidos.iterrows():

    img_path = IMG_DIR / row["imagem"]
    label_path = LAB_DIR / row["label"]

    with Image.open(img_path) as im:
        largura, altura = im.size

    conteudo = label_path.read_text()

    detalhes.append({
        "imagem": row["imagem"],
        "largura": largura,
        "altura": altura,
        "linhas_validas": row["n_bbox_validas"],
        "linhas_totais": row["n_linhas"],
        "erro": row["erros"],
        "conteudo_original": conteudo
    })

df_detalhes = pd.DataFrame(detalhes)

saida = AUDIT_DIR / "labels_invalidos_detalhados.csv"
df_detalhes.to_csv(saida, index=False)

display(df_detalhes)

print("Arquivo salvo:", saida)

,imagem,largura,altura,linhas_validas,linhas_totais,erro,conteudo_original
0,0014.jpg,431,379,0,1,linha 1: bounding box ultrapassa os limites da...,0 0.6765895336567834 0.5874813766313086 0.6498...
1,00177.jpg,1600,900,0,1,linha 1: bounding box ultrapassa os limites da...,0 0.6014113938487238 0.9916444441150722 0.2024...
2,00216.jpg,1424,744,1,2,linha 2: coordenadas fora do intervalo ou dime...,0 0.12721371160051734 0.2214701074479253 0.060...
3,00217.jpg,726,395,0,1,linha 1: bounding box ultrapassa os limites da...,0 0.8350990504842559 0.9593388857198757 0.1436...
4,00285.jpg,646,426,0,1,linha 1: bounding box ultrapassa os limites da...,0 0.2984757094547475 0.8555996568352537 0.4380...
5,00593.png,442,240,0,1,linha 1: coordenadas fora do intervalo ou dime...,0 0.7300035026674543 1.0431454941219767 0.1549...
6,00601.png,330,217,2,3,linha 1: bounding box ultrapassa os limites da...,0 0.621827653511791 0.7906616306062925 0.28257...
7,00605.png,321,253,0,1,linha 1: coordenadas fora do intervalo ou dime...,0 0.3195663266135246 1.1622447637798778 0.2075...
8,00609.png,224,294,1,2,linha 1: bounding box ultrapassa os limites da...,0 0.9225964205840144 0.8450050072297713 0.1569...
9,00620.png,805,469,1,2,linha 2: bounding box ultrapassa os limites da...,0 0.49975690621028407 0.5372167669328611 0.186...


Arquivo salvo: /content/drive/MyDrive/droneshark/auditoria_dataset_v2/labels_invalidos_detalhados.csv


In [23]:
# ============================================================
# VISUALIZAÇÃO DOS 19 LABELS INVÁLIDOS — SEM MODIFICAR ORIGINAIS
# ============================================================

from PIL import Image, ImageDraw, ImageFont
from pathlib import Path
import math
import pandas as pd

VIS_DIR = AUDIT_DIR / "inspecao_visual_labels"
VIS_DIR.mkdir(exist_ok=True)

invalidos = audit_df[
    audit_df["status"] == "invalido_ou_revisar"
].copy()

def desenhar_anotacoes(img_path, label_path):
    img = Image.open(img_path).convert("RGB")
    w, h = img.size
    draw = ImageDraw.Draw(img)

    linhas = label_path.read_text().splitlines()

    for numero, linha in enumerate(linhas, 1):
        partes = linha.split()

        if len(partes) != 5:
            continue

        cls, xc, yc, bw, bh = map(float, partes)

        # Coordenadas originais normalizadas
        x1 = (xc - bw/2) * w
        y1 = (yc - bh/2) * h
        x2 = (xc + bw/2) * w
        y2 = (yc + bh/2) * h

        valida = (
            0 <= xc <= 1 and
            0 <= yc <= 1 and
            0 < bw <= 1 and
            0 < bh <= 1 and
            x1 >= 0 and y1 >= 0 and
            x2 <= w and y2 <= h
        )

        cor = "lime" if valida else "red"

        # Mostra a caixa original, inclusive quando ultrapassa a imagem
        draw.rectangle(
            [x1, y1, x2, y2],
            outline=cor,
            width=3
        )

        draw.text(
            (max(0, min(x1, w-100)), max(0, min(y1, h-20))),
            f"Linha {numero}: {'VALIDA' if valida else 'INVALIDA'}",
            fill=cor
        )

    return img

for _, row in invalidos.iterrows():

    imagem = IMG_DIR / row["imagem"]
    label = LAB_DIR / row["label"]

    if not imagem.exists() or not label.exists():
        continue

    img = desenhar_anotacoes(imagem, label)

    # Redimensionamento para visualização
    img.thumbnail((600, 400))

    nome = Path(row["imagem"]).stem + "_audit.jpg"
    img.save(VIS_DIR / nome, quality=95)

# Criar painéis com 4 imagens por página
arquivos = sorted(VIS_DIR.glob("*_audit.jpg"))

COLS = 2
ROWS = 2
W, H = 620, 450

for pagina, inicio in enumerate(range(0, len(arquivos), COLS*ROWS)):

    subset = arquivos[inicio:inicio+COLS*ROWS]

    painel = Image.new(
        "RGB",
        (COLS*W, ROWS*H),
        "white"
    )

    draw = ImageDraw.Draw(painel)

    for i, arquivo in enumerate(subset):

        imagem = Image.open(arquivo).convert("RGB")
        x = (i % COLS)*W
        y = (i // COLS)*H

        painel.paste(imagem, (x, y+35))

        draw.text(
            (x+5, y+5),
            arquivo.name,
            fill="black"
        )

    painel.save(
        VIS_DIR / f"painel_invalidos_{pagina+1:02d}.jpg",
        quality=95
    )

print("Inspeção visual concluída.")
print("Imagens:", len(arquivos))
print("Painéis:", len(list(VIS_DIR.glob("painel_*.jpg"))))
print("Diretório:", VIS_DIR)

Inspeção visual concluída.
Imagens: 19
Painéis: 5
Diretório: /content/drive/MyDrive/droneshark/auditoria_dataset_v2/inspecao_visual_labels
